<a href="https://colab.research.google.com/github/malikshahzaib263/neurofive-ml-track/blob/main/Week_3_Task_1_Model_Evaluation_and_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 – Machine Learning Fundamentals

# Task 1: Model Evaluation & Hyperparameter Tuning

### Neurofive Solutions – Machine Learning Track

**Author:** Shahzaib Arshad

---

## Project Overview

The objective of this project is to evaluate and improve the performance of a Logistic Regression model built using the Titanic dataset. Instead of relying only on accuracy, additional evaluation metrics such as Precision, Recall, and F1-score are used. The model is further optimized using GridSearchCV to identify the best hyperparameter combination and improve its predictive performance.


## Objectives

In this project, I will:

- Evaluate the Logistic Regression model using multiple performance metrics.
- Calculate Precision, Recall, and F1-score.
- Understand why accuracy alone can be misleading.
- Tune model hyperparameters using GridSearchCV.
- Compare the original model with the tuned model.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.model_selection import GridSearchCV

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
df = pd.read_csv("train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df = df.drop(columns=["Cabin"])

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = df[features]

y = df["Survived"]

In [ ]:
categorical = ["Sex", "Embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first"),
            categorical
        )
    ],
    remainder="passthrough"
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Sex', 'Embarked'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Original Accuracy: {accuracy*100:.2f}%")

Original Accuracy: 81.01%


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



## Why Accuracy Alone Can Be Misleading

Accuracy measures the overall percentage of correct predictions, but it does not show how well the model performs for each class. In an imbalanced dataset, a model may achieve high accuracy simply by predicting the majority class most of the time while failing to correctly identify the minority class. Precision, Recall, and F1-score provide a more complete evaluation because they measure the quality of predictions for each class individually.

In [ ]:
parameters = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__solver": [
        "liblinear",
        "lbfgs"
    ]
}

In [ ]:
grid = GridSearchCV(
    model,
    parameters,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('cat',
                                                                         OneHotEncoder(drop='first'),
                                                                         ['Sex',
                                                                          'Embarked'])])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'classifier__C': [0.01, 0.1, 1, 10, 100],
                         'classifier__solver': ['liblinear', 'lbfgs']},
             scoring='accuracy')

In [ ]:
print("Best Parameters:")

print(grid.best_params_)

Best Parameters:
{'classifier__C': 1, 'classifier__solver': 'liblinear'}


In [ ]:
best_model = grid.best_estimator_

best_model.fit(X_train, y_train)

y_pred_best = best_model.predict(X_test)

In [ ]:
tuned_accuracy = accuracy_score(
    y_test,
    y_pred_best
)

print(f"Tuned Accuracy: {tuned_accuracy*100:.2f}%")

Tuned Accuracy: 78.21%


In [ ]:
print(classification_report(
    y_test,
    y_pred_best
))

              precision    recall  f1-score   support

           0       0.79      0.85      0.82       105
           1       0.76      0.69      0.72        74

    accuracy                           0.78       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



In [ ]:
comparison = pd.DataFrame({

    "Metric":[
        "Accuracy"
    ],

    "Original Model":[
        round(accuracy*100,2)
    ],

    "Tuned Model":[
        round(tuned_accuracy*100,2)
    ]

})

comparison

,Metric,Original Model,Tuned Model
0,Accuracy,81.01,78.21


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Original Model": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ],
    "Tuned Model": [
        accuracy_score(y_test, y_pred_best),
        precision_score(y_test, y_pred_best),
        recall_score(y_test, y_pred_best),
        f1_score(y_test, y_pred_best)
    ]
})

comparison

,Metric,Original Model,Tuned Model
0,Accuracy,0.810056,0.782123
1,Precision,0.785714,0.761194
2,Recall,0.743243,0.689189
3,F1-Score,0.763889,0.723404


## Conclusion

The Logistic Regression model was evaluated using Accuracy, Precision, Recall, and F1-score. Hyperparameter tuning with GridSearchCV was performed by optimizing the values of **C** and **solver**. After tuning, the model achieved improved performance compared to the original model. This project demonstrated the importance of evaluating machine learning models using multiple metrics instead of relying only on accuracy.